# Build the Performance Table
The Goal of this script is tack on performance metrics 

## Read in the data

In [ ]:
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
from pymoo.indicators.igd_plus import IGDPlus
from pymoo.indicators.igd import IGD
from pymoo.indicators.gd import GD
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from matplotlib.animation import FuncAnimation
from IPython.display import HTML

run_file_names = [r"/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2001.pkl", 
                  r"/rdata/ian/pico/paperRuns/finalRuns/global_run_table_2002.pkl"]
years = [2001, 2002]

global_pf_file_name = r"/rdata/ian/pico/paperRuns/global_pf.pkl"

full_run_tabs = [pd.read_pickle(run_file_name) for run_file_name in run_file_names]

full_run_tab = pd.concat(full_run_tabs, axis=0)

del full_run_tabs

global_pf = pd.read_pickle(global_pf_file_name)

global_pf = global_pf.sort_values(by="yield")

gd_ind = GD(global_pf.loc[:,("irr_total", "yield")].values)




## Configuration Definition
What configurations do we want to measure against our baseline? 

In [ ]:
algorithm = ["nsga2",   "pinsga2"]
dm_range =  ["None",    "20to30" ]
pop_size =  [120,       60       ]

configurations = [
    np.all([
        full_run_tab['algorithm'] == algorithm[c], 
        full_run_tab['DM_range']  == dm_range[c],
        full_run_tab['pop_size']  == pop_size[c]
        ], axis=0) 
    for c in range(len(algorithm))  

]


### Performance metric functions
Different ways we can evaluate a given configuration 

#### Final Generational Distance


In [ ]:
def func(df): 
    df['gd_final_gen'] = gd_ind(df.loc[:,('yield','irr_total')].values)
    return df.loc[:,('algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen', 'year')]


def finalGenGD(tab): 

    max_gen = max(tab['gen'])

    # Filter all but the final generations 
    tab = tab[tab["gen"] == max_gen]

    summary = tab.groupby(['algorithm', 'DM_range', 'pop_size', 'run', 'year']).apply(func)

    return summary.drop_duplicates(subset=['algorithm',  'gen', 'DM_range', 'pop_size', 'run', 'gd_final_gen', 'year'])



### Run the analysis 
The goal being that we compress each run into a given metric. So at the end we should have one row per run, with the given metrics calculated for that run

In [ ]:

run_summary  = None 

run_summaries = []

for year in years: 

    for config in configurations:

        config_masks = np.all([config, full_run_tab['year'] == year])

        current_run_summary = finalGenGD(full_run_tab[config])

        run_summaries.append(current_run_summary)

run_summary = pd.concat(run_summaries)

run_summary


## Plot box plots 


In [ ]:
years = set(run_summary['year'])

for year in years:

    year_summary = run_summary[run_summary['year'] == year]
    axes = year_summary.boxplot(by="algorithm", column="gd_final_gen")
    fig = axes.get_figure()
    fig.suptitle(year)



### Build performance table


In [ ]:
years = set(run_summary['year'])
dm_ranges = set(run_summary['DM_range'])

#ds1 = run_summary[run_summary["algorithm"] == "nsga2"]["gd_final_gen"]
#ds2 = run_summary[run_summary["algorithm"] == "pinsga2"]["gd_final_gen"]
#
#wilcoxon(ds1, ds2)

perf_tab_raw = {"year": []}
for dm_range in dm_ranges: 
    perf_tab_raw[dm_range] = []
    if dm_range != "None":
        perf_tab_raw[dm_range + "_perf"] = []


for year in years: 
    perf_tab_raw["year"].append(year)
    for dm_range in dm_ranges: 

        # Pull out the configuration we're looking at 
        select_mask = np.logical_and(run_summary["year"] == year, run_summary["DM_range"] == dm_range)

        # Calculate the median 
        median = np.median(run_summary[select_mask]["gd_final_gen"])
        perf_tab_raw[dm_range].append(median)

        # Do a Wilcoxon rank sum if it's not NSGA-II (dm_range == None)
        if dm_range != "None":

            sample_mask = np.logical_and(
                run_summary["algorithm"] == "pinsga2", 
                run_summary["year"] == year)

            sample_mask_benchmark = np.logical_and(
                run_summary["algorithm"] == "nsga2", 
                run_summary["year"] == year)

            ds1 = run_summary[sample_mask]["gd_final_gen"]

            ds2 = run_summary[sample_mask_benchmark]["gd_final_gen"]
            
            perf_tab_raw[dm_range + "_perf"].append(wilcoxon(ds1,ds2)[1])


pd.DataFrame(perf_tab_raw)

